In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F

# Leitura da tabela limpa da camada Silver
df_silver = spark.read.table("market_prices_db.silver.clean_food_prices")

# Agregação analítica por País, Mercado, Produto e Mês/Ano
df_gold_agg = df_silver.groupBy(
    "adm0_name", "mkt_name", "cm_name", "cur_name", "um_name", "year", "month"
).agg(
    F.avg("price").alias("avg_price"),
    F.max("price").alias("max_price"),
    F.min("price").alias("min_price"),
    F.count("*").alias("total_observations")
)

# Persistência na camada Gold usando Delta Lake
target_table = "market_prices_db.gold.agg_market_prices_summary"
df_gold_agg.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

total_gold = df_gold_agg.count()
print(f"Camada Gold gerada com sucesso. Total de linhas agregadas: {total_gold}")